In [1]:
import tensorflow as tf
import numpy as np
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Lambda
from tensorflow.keras.optimizers import Adam
from keras.layers import Activation, MaxPool2D, Concatenate
from keras.losses import BinaryCrossentropy
from keras.metrics import MeanIoU
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from pathlib import Path
import os
import cv2
import imutils
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset 

In [18]:
path=Path('../../../0_data/Processed/Saeedi/')
path_tot=path/'Images'
path_gt=path/'GT_TE'
path_models=Path('../../../2_models/Kfold_TE')

In [4]:
dim=(256,256)
def load_imgs(path, path_gt=''):
    files=[path/f for f in os.listdir(path)]
    x = np.array([cv2.resize(cv2.imread(str(f),cv2.IMREAD_GRAYSCALE), dim) for f in files])
    if path_gt == '':
        y=None
    else:
        y = np.array([cv2.threshold(cv2.resize(cv2.imread(str(path_gt/(f.stem+' TE_Mask.bmp')),cv2.IMREAD_GRAYSCALE), dim),127,255,cv2.THRESH_BINARY)[1] for f in files])
    return x,y

In [5]:
x,y=load_imgs(path_tot,path_gt)

In [6]:
def std_norm(x):
    mean, std= np.mean(x, axis=0), np.std(x, axis=0) 
    return ((x.astype('float32') - mean) / std , mean, std)

In [7]:
def std_norm_test(x,mean,std):
    return (x.astype('float32') - mean) / std

In [8]:
def data_augmentation(x_train, y_train):
    x_train_augmented, y_train_augmented=[], []
    for k in range(len(x_train)):
        img=x_train[k]
        mask=y_train[k]
        for i in range(37):
            x_train_augmented.append(imutils.rotate(img, angle=10*i))
            y_train_augmented.append(imutils.rotate(mask, angle=10*i))
    return np.array(x_train_augmented), np.array(y_train_augmented)

In [9]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Model

In [10]:
def conv_block(input, num_filters):
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(input)
    x = Dropout(0.4) (x)
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x)
    x = Dropout(0.4) (x)
    return x

#Encoder block: Conv block followed by maxpooling and residual block

def encoder_block(input, num_filters):
    x = conv_block(input, num_filters) # convolutional block
#     x = BatchNormalization()(x)
    input = Conv2D(num_filters, 1, padding="same")(input)
    transfer = tf.math.add(x,input)
    p = MaxPool2D(2)(transfer) #subsampling block  
    p = Activation("relu")(p)
    return x, p  


#Bottleneck block: 4 dilaten convolution layers

def bottleneck_block(input, num_filters):
    x1 = Conv2D(num_filters, 3, dilation_rate=1,  padding="same" )(input)
    x1 =  Dropout(0.4) (x1)
    x2 = Conv2D(num_filters, 3, dilation_rate=2, padding="same" )(x1)
    x2 =  Dropout(0.4) (x2)
    x3 = Conv2D(num_filters, 3, dilation_rate=4, padding="same" )(x2)
    x3 =  Dropout(0.4) (x3)
    x4 = Conv2D(num_filters, 3, dilation_rate=8, padding="same" )(x3)
    x4 =  Dropout(0.4) (x4)
    c = Concatenate(axis=3)([x1, x2, x3, x4])
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(c)
    x = Dropout(0.05) (x)
    return  x


#Decoder block
#skip features gets input from encoder for concatenation

def decoder_block(input, skip_features, num_filters):
    x = UpSampling2D(size=2)(input)
    x = BatchNormalization()(x) 
    x1 = Concatenate(axis=3)([x, skip_features])
      
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x1)
    x = Dropout(0.4) (x)
    x = Conv2D(num_filters, 3, padding="same", activation='relu')(x)
    x = Dropout(0.4) (x)
    x1 = Conv2D(num_filters, 1, padding="same")(x1)
    return tf.math.add(x,x1)

#Build Unet using the blocks
def build_unet(input_shape):
    inputs = Input(input_shape)

    s1, p1 = encoder_block(inputs, 16)
    s2, p2 = encoder_block(p1, 32)
    s3, p3 = encoder_block(p2, 64)
    s4, p4 = encoder_block(p3, 128)

    b1   = bottleneck_block(p4, 128) #Bridge

    d1 = decoder_block(b1, s4, 128)
    d2 = decoder_block(d1, s3, 64)
    d3 = decoder_block(d2, s2, 32)
    d4 = decoder_block(d3, s1, 16)

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)  
    model = Model(inputs, outputs, name="U-Net")
    return model

In [11]:
model= build_unet(input_shape=(256,256,1))

In [12]:
def jaccard_index(y_true, y_pred):
    """ Calculates mean of Jaccard index as a loss function """
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1,2))
    sum_ = tf.reduce_sum(y_true + y_pred, axis=(1,2))
    jac = (intersection) / (sum_ - intersection)
    return tf.reduce_mean(jac)

def my_loss_fn(y_true, y_pred):
    return BinaryCrossentropy(from_logits=True)(y_true, y_pred)-tf.math.log(jaccard_index(y_true, y_pred))

In [13]:
callbacks = [
    EarlyStopping(patience=15),
    ReduceLROnPlateau(factor=0.05, patience=5)]

In [14]:
from tensorflow.keras.utils import Sequence
class DataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.x) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_x = self.x[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_y = self.y[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x, batch_y=shuffle(batch_x, batch_y, random_state=1)
        return batch_x, batch_y

In [15]:
def accuracy(target, prediction):
    true_detec = np.logical_not(np.logical_xor(target, prediction))
    return np.sum(true_detec)/np.sum(np.ones_like(target))

def precision(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/np.sum(prediction)

def recall(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/np.sum(target)

def jaccard(target, prediction):
    intersection = np.logical_and(target, prediction)
    union = np.logical_or(target, prediction)
    return np.sum(intersection) / np.sum(union)

def dice(target, prediction):
    intersection = np.logical_and(target, prediction)
    return 2*np.sum(intersection) / (np.sum(target) + np.sum(prediction))

def metrics(target, prediction):
    return {'accuracy': accuracy(target, prediction),
            'precision': precision(target, prediction),
            'recall': recall(target, prediction),
            'specificity': recall(1-target,1- prediction),
            'jaccard':jaccard(target, prediction),
            'dice': dice(target, prediction)}

def summary_metrics(y_test,predictions,thresh=0.5):
    a,p,r,s,j,d=0.,0.,0.,0.,0.,0.
    n=len(predictions)
    for i in range(n):
        preds= (predictions[i][:,:,0]>=thresh).astype('uint8')
        gt= y_test[i].astype('uint8')
        metricas=metrics(gt,preds)
        a+=metricas['accuracy']
        p+=metricas['precision']
        r+=metricas['recall']
        s+=metricas['specificity']
        j+=metricas['jaccard']
        d+=metricas['dice']
    return {'accuracy': a/n,
            'precision': p/n,
            'recall': r/n,
            'specificity': s/n,
            'jaccard':j/n,
            'dice': d/n}

In [ ]:
fold_no = 1
for train, test in kfold.split(x):
    x_train, y_train = x[train], y[train]
    x_test, y_test = x[test], y[test]
    
    x_train, mean, std = std_norm(x_train)
    x_test = std_norm_test(x_test, mean, std)
    
    x_train, y_train = data_augmentation(x_train, y_train)
    x_train, y_train = shuffle(x_train, y_train, random_state=13)
    
    x_train = x_train.astype("float32") 
    y_train = y_train.astype("float32")/255  

    x_test = x_test.astype("float32") 
    y_test = y_test.astype("float32")/255  
    
    train_gen = DataGenerator(x_train, y_train, 16)
    model= build_unet(input_shape=(256,256,1))
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=my_loss_fn, metrics=[jaccard_index])
    
    print(f'Training for fold {fold_no} ...')
    history = model.fit(train_gen,batch_size=16, epochs=200, callbacks=callbacks, validation_data=(x_test, y_test))
    predictions = model.predict(x_test)
    
    print(f'Score for fold {fold_no}: ' + str(summary_metrics(y_test,predictions, 0.5)))
    model.save('fold'+str(fold_no))    
    fold_no = fold_no + 1

Training for fold 1 ...
Epoch 1/200
518/518 [==============================] - 103s 171ms/step - loss: 1.4405 - jaccard_index: 0.5028 - val_loss: 1.2294 - val_jaccard_index: 0.5757
Epoch 2/200
518/518 [==============================] - 86s 166ms/step - loss: 1.1538 - jaccard_index: 0.6235 - val_loss: 1.1434 - val_jaccard_index: 0.6230
Epoch 3/200
518/518 [==============================] - 86s 167ms/step - loss: 1.1045 - jaccard_index: 0.6526 - val_loss: 1.1124 - val_jaccard_index: 0.6492
Epoch 4/200
518/518 [==============================] - 86s 166ms/step - loss: 1.0760 - jaccard_index: 0.6702 - val_loss: 1.0779 - val_jaccard_index: 0.6716
Epoch 5/200
518/518 [==============================] - 86s 166ms/step - loss: 1.0553 - jaccard_index: 0.6832 - val_loss: 1.0738 - val_jaccard_index: 0.6746
Epoch 6/200
518/518 [==============================] - 86s 166ms/step - loss: 1.0369 - jaccard_index: 0.6950 - val_loss: 1.0510 - val_jaccard_index: 0.6874
Epoch 7/200
518/518 [==================

518/518 [==============================] - 87s 168ms/step - loss: 0.8951 - jaccard_index: 0.7939 - val_loss: 0.9311 - val_jaccard_index: 0.7645
Epoch 61/200
518/518 [==============================] - 87s 169ms/step - loss: 0.8947 - jaccard_index: 0.7942 - val_loss: 0.9316 - val_jaccard_index: 0.7641
Score for fold 2: {'accuracy': 0.96888671875, 'precision': 0.8175814382729921, 'recall': 0.9290798650731182, 'specificity': 0.9737572576927931, 'jaccard': 0.7688751993704139, 'dice': 0.8678481322074179}
INFO:tensorflow:Assets written to: fold2\assets
Training for fold 3 ...
Epoch 1/200
518/518 [==============================] - 90s 167ms/step - loss: 1.4474 - jaccard_index: 0.4995 - val_loss: 1.2507 - val_jaccard_index: 0.5720
Epoch 2/200
518/518 [==============================] - 86s 167ms/step - loss: 1.1644 - jaccard_index: 0.6173 - val_loss: 1.1390 - val_jaccard_index: 0.6366
Epoch 3/200
518/518 [==============================] - 86s 166ms/step - loss: 1.1099 - jaccard_index: 0.6493 - v

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



518/518 [==============================] - 841887s 1628s/step - loss: 0.9035 - jaccard_index: 0.7876 - val_loss: 0.9695 - val_jaccard_index: 0.7452
Epoch 40/200
518/518 [==============================] - 132s 255ms/step - loss: 0.9030 - jaccard_index: 0.7881 - val_loss: 0.9695 - val_jaccard_index: 0.7453
Epoch 41/200
518/518 [==============================] - 126s 243ms/step - loss: 0.9034 - jaccard_index: 0.7877 - val_loss: 0.9697 - val_jaccard_index: 0.7450
Epoch 42/200
518/518 [==============================] - 126s 243ms/step - loss: 0.9035 - jaccard_index: 0.7877 - val_loss: 0.9700 - val_jaccard_index: 0.7445
Epoch 43/200
518/518 [==============================] - 126s 243ms/step - loss: 0.9033 - jaccard_index: 0.7878 - val_loss: 0.9694 - val_jaccard_index: 0.7451
Score for fold 3: {'accuracy': 0.9632464599609375, 'precision': 0.8288016807097739, 'recall': 0.8725503861984352, 'specificity': 0.9752966716903356, 'jaccard': 0.7404144993814008, 'dice': 0.8455100084580862}
INFO:tensorf

Epoch 14/200
518/518 [==============================] - 86s 165ms/step - loss: 0.9702 - jaccard_index: 0.7400 - val_loss: 0.9583 - val_jaccard_index: 0.7543
Epoch 15/200
518/518 [==============================] - 88s 169ms/step - loss: 0.9665 - jaccard_index: 0.7426 - val_loss: 0.9610 - val_jaccard_index: 0.7527
Epoch 16/200
518/518 [==============================] - 88s 169ms/step - loss: 0.9626 - jaccard_index: 0.7453 - val_loss: 0.9714 - val_jaccard_index: 0.7472
Epoch 17/200
518/518 [==============================] - 85s 165ms/step - loss: 0.9587 - jaccard_index: 0.7480 - val_loss: 0.9743 - val_jaccard_index: 0.7457
Epoch 18/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9551 - jaccard_index: 0.7506 - val_loss: 0.9929 - val_jaccard_index: 0.7323
Epoch 19/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9517 - jaccard_index: 0.7530 - val_loss: 0.9787 - val_jaccard_index: 0.7438
Epoch 20/200
518/518 [==============================] - 86

Epoch 31/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9198 - jaccard_index: 0.7756 - val_loss: 0.9770 - val_jaccard_index: 0.7423
Epoch 32/200
518/518 [==============================] - 85s 163ms/step - loss: 0.9127 - jaccard_index: 0.7808 - val_loss: 0.9486 - val_jaccard_index: 0.7624
Epoch 33/200
518/518 [==============================] - 85s 163ms/step - loss: 0.9109 - jaccard_index: 0.7821 - val_loss: 0.9567 - val_jaccard_index: 0.7563
Epoch 34/200
518/518 [==============================] - 85s 163ms/step - loss: 0.9101 - jaccard_index: 0.7827 - val_loss: 0.9546 - val_jaccard_index: 0.7573
Epoch 35/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9097 - jaccard_index: 0.7829 - val_loss: 0.9541 - val_jaccard_index: 0.7576
Epoch 36/200
518/518 [==============================] - 85s 163ms/step - loss: 0.9088 - jaccard_index: 0.7836 - val_loss: 0.9558 - val_jaccard_index: 0.7563
Epoch 37/200
518/518 [==============================] - 85

Epoch 30/200
518/518 [==============================] - 85s 165ms/step - loss: 0.9331 - jaccard_index: 0.7659 - val_loss: 0.9849 - val_jaccard_index: 0.7380
Epoch 31/200
518/518 [==============================] - 85s 165ms/step - loss: 0.9324 - jaccard_index: 0.7664 - val_loss: 0.9838 - val_jaccard_index: 0.7382
Epoch 32/200
518/518 [==============================] - 88s 169ms/step - loss: 0.9319 - jaccard_index: 0.7667 - val_loss: 0.9830 - val_jaccard_index: 0.7391
Epoch 33/200
518/518 [==============================] - 88s 170ms/step - loss: 0.9315 - jaccard_index: 0.7670 - val_loss: 0.9824 - val_jaccard_index: 0.7395
Epoch 34/200
518/518 [==============================] - 85s 164ms/step - loss: 0.9309 - jaccard_index: 0.7674 - val_loss: 0.9826 - val_jaccard_index: 0.7394
Epoch 35/200
518/518 [==============================] - 85s 165ms/step - loss: 0.9306 - jaccard_index: 0.7677 - val_loss: 0.9820 - val_jaccard_index: 0.7397
Epoch 36/200
518/518 [==============================] - 85

# Metrics

In [19]:
fold_no = 1
m1=[]
for train, test in kfold.split(x):
    x_train, y_train = x[train], y[train]
    x_test, y_test = x[test], y[test]
    
    x_train, mean, std = std_norm(x_train)
    x_test = std_norm_test(x_test, mean, std)

    x_test = x_test.astype("float32") 
    y_test = y_test.astype("float32")/255 
    
    model= tf.keras.models.load_model(path_models/'fold{}'.format(fold_no),custom_objects={'jaccard_index':jaccard_index, 'my_loss_fn':my_loss_fn})
    predictions = model.predict(x_test)
    print(summary_metrics(y_test,predictions, 0.5))
    m1.append(summary_metrics(y_test,predictions, 0.5))
    fold_no = fold_no + 1

{'accuracy': 0.9642364501953125, 'precision': 0.8070241684098352, 'recall': 0.9008058428526138, 'specificity': 0.9741016011194898, 'jaccard': 0.7354296636486783, 'dice': 0.8454264057547585}
{'accuracy': 0.968887939453125, 'precision': 0.8175822484167584, 'recall': 0.9290903172066001, 'specificity': 0.9737572888206719, 'jaccard': 0.7688839556723821, 'dice': 0.8678534514231431}
{'accuracy': 0.963248291015625, 'precision': 0.8288123808666598, 'recall': 0.8725562088243479, 'specificity': 0.9752981044260064, 'jaccard': 0.7404288595640363, 'dice': 0.8455184455598642}
{'accuracy': 0.966134033203125, 'precision': 0.8211737945445359, 'recall': 0.8911932938683161, 'specificity': 0.9756742158569541, 'jaccard': 0.7452074100729347, 'dice': 0.850890928253758}
{'accuracy': 0.95748046875, 'precision': 0.809364076596784, 'recall': 0.8663679525897503, 'specificity': 0.972512855369189, 'jaccard': 0.7169514385505991, 'dice': 0.8284998342911831}
{'accuracy': 0.96606201171875, 'precision': 0.789961542842808

In [20]:
accuracy=[el['accuracy'] for el in m1]
precision=[el['precision'] for el in m1]
recall=[el['recall'] for el in m1]
specificity=[el['specificity'] for el in m1]
jaccard=[el['jaccard'] for el in m1]
dice=[el['dice'] for el in m1]

In [21]:
np.mean(accuracy), np.std(accuracy)

(0.9612199427286783, 0.006052722076176182)

In [22]:
np.mean(precision), np.std(precision)

(0.7909293171381437, 0.02961554988303093)

In [23]:
np.mean(recall), np.std(recall)

(0.8958929419775895, 0.04671924460203954)

In [24]:
np.mean(specificity), np.std(specificity)

(0.9700524796255351, 0.005672048271704256)

In [25]:
np.mean(jaccard), np.std(jaccard)

(0.7220413067477289, 0.03901050161123595)

In [26]:
np.mean(dice), np.std(dice)

(0.8328625685192378, 0.0321602447858363)